# Member B — Behavior + Online Retail II Data Transformation

Transforms two datasets into canonical-schema CSVs (`backend/schema.sql`):

1. **eCommerce behavior data from multi category store** (2019-Oct/Nov) — feeds Predicted Purchase Intent (Segment 1) and Cart Abandoners (Segment 9). Target table: `events` (+ the `customers`/`products` dimension rows its FKs require).
2. **Online Retail II** — feeds Future High-Value / CLV (Segment 2). Targets: `customers`, `products`, `orders`, `order_items`.

**Spec:** `BEHAVIOR_AND_RETAIL.md` — mappings with citations, gaps, and open questions.
**Logic:** `common.py` (id namespace + shared timeline), `behavior.py` (streaming events), `retail.py` (orders/items), `validate.py` (constraint checks).

This notebook orchestrates those modules — the transformation itself is not duplicated here, so there is one source of truth.

Raw files: `../raw/behavior/` + `../raw/retail/` · Output: `../processed/member_b/`

> The behavior source is ~115M rows, so it streams chunk-by-chunk straight to `events.csv` — never held in memory. Run with `USE_FIXTURES = True` for a fast dry run on the synthetic fixtures.

In [ ]:
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd

import behavior
import retail
from common import SHARED_WINDOW_START, uid
from validate import validate

USE_FIXTURES = False   # True = synthetic fixtures (seconds), False = real Kaggle data

if USE_FIXTURES:
    BEHAVIOR_RAW = Path('fixtures/behavior')
    RETAIL_RAW = Path('fixtures/retail')
else:
    BEHAVIOR_RAW = Path('../raw/behavior')
    RETAIL_RAW = Path('../raw/retail')

OUT_DIR = Path('../processed/member_b')
CHUNK_SIZE = 1_000_000
EVENTS_READBACK_MAX_BYTES = 200_000_000  # read events.csv back for validation only when small
STORE_ID = uid('stores', 'member-b-demo')   # deterministic tenant placeholder

print(f'behavior : {BEHAVIOR_RAW}')
print(f'retail   : {RETAIL_RAW}')
print(f'output   : {OUT_DIR}')
print(f'timeline : earliest date in each source -> {SHARED_WINDOW_START}')

In [ ]:
# ---- behavior: one streaming pass to find the dataset's earliest event ----
behavior_paths = behavior.resolve_raw_files(BEHAVIOR_RAW)
for p in behavior_paths:
    print(f'  {p}')

started = time.time()
offset = behavior.compute_offset(behavior_paths, CHUNK_SIZE)
print(f'  shared timeline offset: {offset} days '
      f'(earliest event -> {SHARED_WINDOW_START}, {time.time() - started:.1f}s)')

In [ ]:
# ---- behavior: stream events straight to disk; dimensions stay in memory ----
OUT_DIR.mkdir(parents=True, exist_ok=True)
events_path = OUT_DIR / 'events.csv'
if events_path.exists():
    events_path.unlink()

b_result = behavior.TransformResult()
started = time.time()
b_frames = behavior.transform_stream(
    behavior_paths, STORE_ID, b_result, events_path, offset, CHUNK_SIZE
)
print(f'  {b_result.events_written:,} events written to {events_path.name} '
      f'in {time.time() - started:.1f}s')
for table, frame in b_frames.items():
    print(f'  {table:<20} {len(frame):>10,} rows')

In [ ]:
# ---- Online Retail II (real dates -> same shared timeline) ----------------
r_raw = retail.load_raw(RETAIL_RAW)
for name, df in r_raw.items():
    print(f'  {name:<32} {len(df):>12,} rows')

r_result = retail.transform(r_raw, STORE_ID)
for table, frame in r_result.frames.items():
    print(f'  {table:<20} {len(frame):>10,} rows')

In [ ]:
# ---- merge the shared dimensions under one tenant -------------------------
# Both datasets are namespaced (evb: / ret:), so ids can never collide inside
# UNIQUE (store_id, source_customer_id) / (store_id, source_product_id).
frames = {
    'stores': (
        pd.concat([b_frames['stores'], r_result.frames['stores']], ignore_index=True)
        .drop_duplicates(subset='id')
    ),
    'customers': (
        pd.concat([b_frames['customers'], r_result.frames['customers']], ignore_index=True)
        .drop_duplicates(subset=['store_id', 'source_customer_id'])
    ),
    'products': (
        pd.concat([b_frames['products'], r_result.frames['products']], ignore_index=True)
        .drop_duplicates(subset=['store_id', 'source_product_id'])
    ),
    'orders': r_result.frames['orders'],
    'order_items': r_result.frames['order_items'],
}
print({t: len(f) for t, f in frames.items()})

In [ ]:
# ---- validation (NOT NULL / PK / UNIQUE / FK / calendar) -----------------
# events.csv is too large to hold for the real dataset; read it back only
# when small (fixtures / smoke runs). Otherwise its integrity holds by
# construction — customers/products were built from the same event rows.
events_size = events_path.stat().st_size if events_path.exists() else 0
readback = events_size <= EVENTS_READBACK_MAX_BYTES
if readback:
    frames['events'] = pd.read_csv(events_path, dtype=str, low_memory=False)
else:
    print(f'events.csv is {events_size / 1e9:.1f} GB — skipping in-memory readback')

raw_counts = {**b_result.raw_counts, **r_result.raw_counts}
dropped = {}
for k, v in {**b_result.dropped, **r_result.dropped}.items():
    dropped[k] = dropped.get(k, 0) + v
dropped_by_source = {}
for k, v in {**b_result.dropped_by_source, **r_result.dropped_by_source}.items():
    dropped_by_source[k] = dropped_by_source.get(k, 0) + v

columns = dict(behavior.COLUMNS)
columns['orders'] = retail.COLUMNS['orders']
columns['order_items'] = retail.COLUMNS['order_items']
if not readback:
    columns.pop('events')

report = validate(
    {t: f for t, f in frames.items() if t in columns},
    raw_counts, dropped, dropped_by_source, anchor=None, columns=columns,
)
print(report.render())

if dropped:
    print('\nDROPPED ROWS (source rows that could not be placed):')
    for reason, count in sorted(dropped.items(), key=lambda kv: -kv[1]):
        print(f'  {count:>10,}  {reason}')

notes = list(dict.fromkeys(b_result.notes + r_result.notes))
if notes:
    print('\nNOTES / UNMAPPED:')
    for note in notes:
        print(f'  - {note}')

In [ ]:
# ---- write canonical CSVs + audit.json (only when validation passes) -----
LOAD_ORDER = ['stores', 'customers', 'products', 'orders', 'order_items', 'events']

if not report.ok:
    print('VALIDATION FAILED — nothing written. Fix the transform and re-run.')
else:
    for table in LOAD_ORDER:
        if table in frames:
            frames[table].to_csv(OUT_DIR / f'{table}.csv', index=False, na_rep='')

    audit = {
        'member': 'B',
        'store_id_placeholder': STORE_ID,
        'behavior': {
            'sources': [str(p) for p in behavior_paths],
            'shared_timeline_offset_days': offset,
            'events_written': b_result.events_written,
            'events_violations': b_result.events_violations,
            'raw_counts': b_result.raw_counts,
            'dropped': b_result.dropped,
            'dropped_by_source': b_result.dropped_by_source,
            'notes': b_result.notes,
        },
        'retail': {
            'raw_counts': r_result.raw_counts,
            'dropped': r_result.dropped,
            'dropped_by_source': r_result.dropped_by_source,
            'notes': r_result.notes,
        },
        'emitted_counts': {t: int(len(frames[t])) for t in frames},
        'events_readback_validated': readback,
        'validation': {'ok': report.ok,
                       'failures': report.failures,
                       'warnings': report.warnings},
        'load_order': LOAD_ORDER,
    }
    (OUT_DIR / 'audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    print(f'Wrote {len(LOAD_ORDER)} canonical CSVs + audit.json to {OUT_DIR}')
    print('Load order (FK-safe): ' + ' -> '.join(LOAD_ORDER))